# EgoExo-Fitness 微调：动作质量分 + 可解释关键点验证

任务：**单动作视频 → 1~5 分质量分（有序回归） + 逐条技术关键点达标/不达标（多标签二分类）**。

> 论文最强 GEV 基线 F1 = **0.5439**。稳过 0.55 就是增量，别把目标定在 0.9。

**执行顺序：从上往下。第 6 格（smoke）不要跳过** —— 它用合成数据验证全链路，不需要下载任何东西。


## 0 环境自检


In [ ]:
!nvidia-smi | head -12
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('python', sys.version.split()[0])
print()
print('⚠️ T4 是 Turing 架构，不支持 bf16。代码里已用 fp16 + GradScaler。')


## 1 拉代码

仓库是 public，Colab 侧**不需要任何 GitHub 凭证**。


In [ ]:
import os
REPO = 'https://github.com/introspectDaily/egoexo-fitness-aqa.git'
PROJ = '/content/egoexo-fitness-aqa'

if os.path.exists(PROJ):
    !cd {PROJ} && git pull --ff-only
else:
    !git clone -q {REPO} {PROJ}
%cd {PROJ}
!git log --oneline -3 && ls


## 2 装依赖

torch/torchvision 用 Colab 预装的，**不重装**（重装大概率会装错 CUDA 版本）。


In [ ]:
!pip install -q -e . 2>&1 | tail -3
import egoexo; print('egoexo', egoexo.__version__, '->', egoexo.__file__)


## 3 凭证自检（全程掩码，不会打印明文）

两个 Secret 都要在左侧 🔑 面板里**把该行的 Notebook access 开关打开**（默认是关的）。

> 注意：Secret 配置好后**必须重启 runtime** 才能生效（这是 Colab 的已知行为）。


In [ ]:
from egoexo.secrets import describe, hf_login
print('凭证状态:', describe())
tok = hf_login(quiet=False)   # 只写入 ~/.cache/huggingface/token，不进环境变量
print('HF token 可用' if tok else '未配置 HF token（gated 数据会 401）')


## 4 冒烟测试：合成数据跑通全链路

**这一步不要跳。** 用随机张量把「建表→前向→反向→评估→存档」跑一遍，
过了它，后面出问题就一定是数据/环境问题。


In [ ]:
!python -m egoexo.cli smoke --epochs 3 --frames 16 --out runs/smoke


---
# 数据准备


## 5 下载标注（6MB）并看数据概览

这一步会顺便算 **Krippendorff's α**（标注者间一致性）。
如果 α < 0.6，说明标注规范本身有分歧，先别急着调模型。


In [ ]:
!python scripts/download_data.py --annotations-only
!python -m egoexo.cli stats --raw-dir data/raw_annotations --out data/dataset_stats.json


## 6 下载视觉特征（6.4GB，约 5~10 分钟）

特征在 HF 上是 2 个分片，脚本会自动合并 + 解压 + 清理中间产物。

> 断点续传：`hf_hub_download` 自带，中断了重跑这一格即可。


In [ ]:
!python scripts/download_data.py
!du -sh data/features_open


## 7 探测特征文件结构

确认 `.pth` 里张量的形状符合预期（(T, 512) 或可规整成该形状）。


In [ ]:
!python -m egoexo.cli probe --feat-root data/features_open --max-files 3


## 8 预抽取：把每个动作的时间窗切成定长帧

原始是每个 (record, view) 一个 ~16MB 的 `.pth`（整段视频逐帧特征）。
预抽取后变成一个大 `(N, T, 512)` float16 数组（约 176MB），之后训练是纯内存读取。

`--crop-ratio 0.8` 表示只取时间窗**居中 80%**：动作的首尾（走近、站定、喘气）对质量分是纯噪声。


In [ ]:
!python -m egoexo.cli extract \
    --raw-dir data/raw_annotations \
    --feat-root data/features_open \
    --out data/precomputed \
    --num-frames 32 \
    --crop-ratio 0.8
!ls -lh data/precomputed && du -sh data


---
# 训练


## 9 单折快速验证（先确认能学动）

先跑 1 折 15 epoch，看 SROCC 是不是明显高于 0（随机基线 SROCC≈0）。
如果这一格 SROCC 就已经接近 0.6，说明数据里有信号，再做 5 折。


In [ ]:
!python -m egoexo.cli train \
    --precomputed data/precomputed --out runs/quick \
    --folds 2 --epochs 15


## 10 正式训练：5 折 GroupKFold

划分按 `record_id` 分组（等价于按人分组），**无泄漏**，每次训练都有断言检查。

关键超参：`--w-align 0.7` 是论文的跨视角对齐权重（消融时改成 0 对比）。


In [ ]:
!python -m egoexo.cli train \
    --precomputed data/precomputed --out runs/exp1 \
    --folds 5 --epochs 40 --batch-size 32 --lr 3e-4 \
    --dropout 0.2 --score-noise 0.15 --ema-decay 0.999 \
    --w-score 1.0 --w-keypoint 1.0 --w-action 0.3 --w-align 0.7


## 11 看结果


In [ ]:
import json, pandas as pd
s = json.load(open('runs/exp1/cv_summary.json'))
rows = [{**{'fold': i}, **f} for i, f in enumerate(s['per_fold'])]
df = pd.DataFrame(rows)[['fold','srocc','plcc','mae','acc1','kp_f1','kp_f1_best','srocc_ego','srocc_exo']]
print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print()
print('均值:', {k: round(v,4) for k, v in s['mean'].items()})
print()
print('论文 GEVFormer 的 GEV F1 = 0.5439 —— 对比 kp_f1_best 看是否超过')


## 12 消融：跨视角对齐到底有没有用

论文的核心发现之一：**简单混训 ego+exo 不涨点，只有显式对齐才有效**。
把 `--w-align` 设成 0 重跑一次，直接对比 ego / exo 的 SROCC 和 F1。


In [ ]:
# !python -m egoexo.cli train --precomputed data/precomputed --out runs/no_align --folds 5 --epochs 40 --w-align 0
# 只训 ego 视角（对照「单视角模型」）
# !python -m egoexo.cli train --precomputed data/precomputed --out runs/ego_only --folds 5 --epochs 40 --only-views ego_l,ego_m,ego_r


## 13 保存产物到 Google Drive（可选，防 runtime 被回收）


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/egoexo_runs && cp -r runs/exp1 /content/drive/MyDrive/egoexo_runs/
